# Recursive Language Models: The Prompt as an Environment

The full code of [Recursive Language Models: The Prompt as an Environment](https://residual-stream.lvtan71.workers.dev/posts/recursive-language-models/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/recursive-language-models) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/recursive-language-models.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 7 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 7 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.18.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "recursive-language-models"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/recursive-language-models


In [2]:
!tar xzf blog-73b66ae.tar.gz

## `rlm.py`

A miniature recursive language model on Qwen2.5-3B-Instruct: the corpus sits in a REPL as `context`, the model writes code to read it and can call `llm_query` on a piece. Ten checkable tasks on the blog's posts (five planted needles, five counts), against reading the first 8,000 tokens directly and a fixed program making the same kind of sub-calls. Saves every turn and sub-call to `rlm_transcripts.json`.

In [3]:
!{sys.executable} rlm.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (255743 > 131072). Running this sequence through the model will result in indexing errors


corpus: 66 posts, 991,185 characters (~255,743 tokens)

task         truth           direct              RLM  calls   sub            fixed program  calls  minutes: direct, RLM, fixed


needle   ochre-876 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.1, 0.2, 0.5


needle   amber-365 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.1, 0.2, 0.5


needle   ochre-514 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.1, 0.2, 0.5


...

count           36         wrong: 3            right      2     0                right: 36     73  0.0, 0.0, 0.5


count           34         wrong: 3       wrong: 147      3     0                wrong: 32     71  0.0, 0.1, 0.5


count           14         wrong: 3            right      3     0                wrong: 12     77  0.0, 0.0, 0.5


count           14         wrong: 2       wrong: 144      3     0                wrong: 10     80  0.0, 0.1, 0.5


count           37         wrong: 3          wrong:       8     0                wrong: 25     75  0.0, 0.1, 0.5

right: direct 0 of 10, RLM 2 of 10, fixed program 6 of 10


The post printed:

```text
corpus: 66 posts, 991,185 characters (~255,743 tokens)

task         truth           direct              RLM  calls   sub            fixed program  calls  minutes: direct, RLM, fixed
needle   ochre-876 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.4, 0.5, 9.3
needle   amber-365 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.4, 0.5, 8.6
needle   ochre-514 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.4, 0.5, 8.6
needle   ochre-466 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.4, 0.5, 8.6
needle  cobalt-388 wrong: Based on           wrong:       5     2   right: 2 hits, 1 codes     53  0.4, 0.5, 8.6
count           36         wrong: 3            right      2     0                right: 36     73  0.3, 0.1, 8.2
count           34         wrong: 3       wrong: 147      3     0                wrong: 32     71  0.3, 0.2, 8.1
count           14         wrong: 3            right      3     0                wrong: 12     77  0.3, 0.1, 8.3
count           14         wrong: 2       wrong: 144      3     0                wrong: 10     80  0.3, 0.2, 8.4
count           37         wrong: 3          wrong:       8     0                wrong: 25     75  0.3, 0.3, 8.2

right: direct 0 of 10, RLM 2 of 10, fixed program 6 of 10
```